# Publish a private-network Foundry agent to Microsoft 365 & Teams

This notebook walks through the same 5-step REST flow the `publish-agent-m365` skill automates, one step per cell, so you can watch each request and its result.

Microsoft 365 **does not support private connectivity** to agents — it requires the agent's **Activity Protocol** endpoint to be publicly routable. This flow opens a scoped, **source-IP-filtered** public exception on *only* the Activity Protocol route (service-managed by Foundry). Responses, MCP, A2A, management APIs, and the Foundry→Fabric data path stay private.

| Step | What it does |
| --- | --- |
| 1 | Resolve the agent identity (`instance_identity.client_id`) + tenant ID |
| 2 | Create (or reuse) the Azure **Bot Service** (PNA disabled, Teams channel) |
| 3 | PATCH `enable_m365_public_endpoint` *(only when the project is locked down)* |
| 4 | POST the Microsoft 365 publish API |
| 5 | Verify in Teams / M365 Copilot |

> The notebook **reuses the skill's tested functions** (`scripts/publish_agent_m365.py`) rather than re-implementing the REST calls, so behaviour matches the CLI exactly.

Reference: [Publish agents to M365 & Teams via REST](https://learn.microsoft.com/azure/foundry/agents/how-to/publish-copilot-virtual-network?view=foundry)

## Prerequisites

1. **Roles:** `Foundry User` on the project **+** `Azure Bot Service Contributor` (or Contributor/Owner) on the resource group.
2. **Run from inside the VNet** (VM / VPN / ExpressRoute) so the management REST calls can reach the project's private endpoint.
3. Target a **persistent, published** agent version (stable name/version), not an ephemeral test one.
4. Register the provider and sign in (run the setup cell below).

In [ ]:
# One-time setup: dependencies, provider registration, and az login.
# Safe to re-run; skip lines you have already done.
%pip install -q azure-identity requests
!az provider register --namespace Microsoft.BotService
!az login
!az account show --query "{subscription:name, id:id, tenant:tenantId}" -o table

## Configure

Fill in the values below. `DRY_RUN = True` prints every request **without mutating anything** — flip it to `False` only when you are ready to actually create the bot / PATCH / publish.

In [ ]:
import importlib.util
from argparse import Namespace
from pathlib import Path

# ---- Load the skill's script as a module (same folder as this notebook) ----
SCRIPT = Path.cwd() / "scripts" / "publish_agent_m365.py"
assert SCRIPT.exists(), f"Cannot find {SCRIPT} — run this notebook from the skill folder."
_spec = importlib.util.spec_from_file_location("publish_agent_m365", SCRIPT)
mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(mod)

# ---- Toggle this to actually make changes ----
DRY_RUN = True

# ---- Your values ----
cfg = Namespace(
    endpoint="https://<res>.services.ai.azure.com/api/projects/<proj>",  # project endpoint
    resource_group="<your-resource-group>",   # RG that contains the Foundry resource
    agent_name="<your-agent-name>",           # persistent, published agent
    bot_name="<new-bot-name>",                 # Azure Bot Service to create (if not reusing)
    agent_display_name="<Display Name>",       # shown in Teams/M365 (blank -> agent name)
    publish_scope="Shared",                    # Shared | Personal | Tenant
    app_version="1.0.0",                        # increment to re-publish
    short_description="<short description>",
    full_description="<full description>",
    developer_name="<developer name>",
    developer_website_url="https://<developer-website>",
    privacy_url="https://<privacy-url>",
    terms_of_use_url="https://<terms-of-use-url>",
    # ---- flow control (defaults mirror the CLI) ----
    bot_arm_id=None,          # set to reuse a specific existing Bot Service ARM ID
    skip_bicep=False,
    force_bicep=False,        # True = always create a new bot even if one exists
    force_patch=False,        # True = always PATCH even on a public project
    no_network_check=False,
    check_bot=False,
    scan_resource_group=False,  # True = limit bot search to the RG (default: whole subscription)
    dry_run=DRY_RUN,
)
cfg.endpoint = cfg.endpoint.rstrip("/")

# Acquire an ai.azure.com token from your `az login` session.
from azure.identity import AzureCliCredential
credential = AzureCliCredential()
token = mod.get_token(credential) if not cfg.dry_run else "<token>"
print(f"DRY_RUN={cfg.dry_run}  agent={cfg.agent_name}  scope={cfg.publish_scope}")

## (Optional) Check for an existing Bot Service first

Read-only. Reports the agent's current M365 endpoint config, the Foundry account's `publicNetworkAccess` (so you know whether the PATCH is even needed), and scans the **whole subscription** for a Bot Service already wired to this agent.

In [ ]:
mod.check_association(cfg, token)

## Step 1 — Get the agent identity + tenant ID

`GET {endpoint}/agents/{name}` → `instance_identity.client_id`; tenant via `az account show`.

In [ ]:
client_id, tenant_id, agent_body = mod.step_identity(cfg, token)
client_id, tenant_id

## Step 2 — Create (or reuse) the Azure Bot Service

Deploys `scripts/bot_service.bicep` (public network access disabled + Teams channel) into your resource group. If a bot already associated with this agent is found, it is **reused** instead of creating a duplicate (override with `cfg.force_bicep = True`).

In [ ]:
bot_arm_id = cfg.bot_arm_id
if not cfg.skip_bicep:
    bot_arm_id = mod.step_bicep(cfg, client_id, tenant_id)
bot_arm_id

## Step 3 — Enable the public Activity Protocol endpoint *(locked-down projects only)*

This checks the Foundry account's `publicNetworkAccess`. If it is **Disabled/restricted**, it PATCHes `activity.enable_m365_public_endpoint: true` (re-sending `responses`, `Entra`, and the scope-matched Bot Service scheme, since the PATCH **replaces** protocol/auth config). On a **public** project it auto-skips (Foundry's one-click publish would work). Force it with `cfg.force_patch = True`.

> The source-IP filtering is **service-managed by Foundry** — you only set this boolean; Microsoft maintains the allowed Bot Service / M365 source ranges. Auth (`Entra` / `BotServiceRbac` / `BotServiceTenant`) still applies on top.

In [ ]:
run_patch = True
if not cfg.force_patch and not cfg.no_network_check:
    pna, restricted, resolved = mod.network_posture(cfg)
    if resolved and not restricted:
        print(f"Skipping PATCH: publicNetworkAccess='{pna}' (unrestricted) — not required for public projects.")
        run_patch = False
    elif resolved:
        print(f"publicNetworkAccess='{pna}' (restricted) — PATCH required.")
    else:
        print("Could not determine network posture — running PATCH (safe default).")

if run_patch:
    mod.step_patch(cfg, token)

## Step 4 — Publish to Microsoft 365

`POST {endpoint}/agents/{name}/microsoft365/publish` → returns the published `titleId`.

**Scope ↔ auth** (the PATCH scheme is picked automatically to match): `Shared`/`Personal` → `BotServiceRbac` (you only, share by link, no admin approval); `Tenant` → `BotServiceTenant` (whole tenant, after M365 admin approval).

> Re-publishing requires a **new `appVersion`** (digits/periods only, cannot start with `0`) — a duplicate returns a *version already exists* error. Do **not** put secrets in any metadata field; they are user-visible.

In [ ]:
mod.step_publish(cfg, token, bot_arm_id)

## Step 5 — Verify

1. Open the M365 / Teams agent store: `Shared` → **Your agents**; `Tenant` → **Built by your org** (after admin approval).
2. Start a conversation and send a message.
3. Confirm the agent replies — that validates end-to-end channel delivery.

### Notes
- **Data stays private:** only inbound Teams/M365 message delivery uses the source-IP-filtered public route; the agent's Foundry→Fabric query still traverses Private Link. Outbound/egress is unchanged.
- **Limitations:** no file uploads / image generation in M365 (works in Teams); no streaming responses or citations for published agents.
- Prefer the CLI? The same flow runs as: `python scripts/publish_agent_m365.py` (add `--dry-run`, `--check-bot`, or `--only patch,publish` as needed).